# Check Recent Video Processing Status

This notebook checks if newly uploaded Keith Foskey videos were successfully processed and displays error logs if any failures occurred.

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import pandas as pd

# Load environment variables
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in environment variables")

# Create database connection
engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

print("✓ Database connection established")

✓ Database connection established


## Check Recent Videos from Playlist

Query the most recent videos ordered by publish date.

In [2]:
def check_recent_videos(days_back=7, limit=20):
    """Check videos from the last N days."""
    with SessionLocal() as session:
        # Get recent videos
        query = text("""
            SELECT 
                youtube_id,
                title,
                published_at,
                status,
                processed_at,
                error,
                created_at
            FROM videos
            ORDER BY published_at DESC NULLS LAST, created_at DESC
            LIMIT :limit
        """)
        
        result = session.execute(query, {"limit": limit})
        rows = result.fetchall()
        
        if not rows:
            print("No videos found in database")
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(rows, columns=[
            'youtube_id', 'title', 'published_at', 'status', 
            'processed_at', 'error', 'created_at'
        ])
        
        return df

# Get recent videos
recent_videos = check_recent_videos(days_back=30, limit=20)

if recent_videos is not None:
    print(f"Found {len(recent_videos)} videos\n")
    
    # Display summary by status
    print("Status Summary:")
    print(recent_videos['status'].value_counts())
    print("\n" + "="*80 + "\n")
    
    # Show all videos
    display(recent_videos[['youtube_id', 'title', 'published_at', 'status', 'processed_at']])

Found 20 videos

Status Summary:
status
processed    20
Name: count, dtype: int64




,youtube_id,title,published_at,status,processed_at
0,JvX7va2U148,YourCalvinist LIVE with Keith & Jennifer Foske...,2026-03-04 04:31:07+00:00,processed,2026-03-05 07:08:13.858517+00:00
1,kMgr2QoClxg,Marriage Matters with Guests David and Inga-Li...,2026-02-28 15:34:12+00:00,processed,2026-03-01 07:05:54.863437+00:00
2,pRVSnDoj8tE,Your Calvinist Live with Keith & Jennifer Foskey,2026-02-25 04:14:27+00:00,processed,2026-02-27 21:30:09.527098+00:00
3,QswIfBUvJgY,YourCalvinist Live with Keith & Jennifer Foske...,2026-02-18 04:09:38+00:00,processed,2026-02-27 21:25:51.721957+00:00
4,2jokE0SLgB0,YourCalvinist LIVE with Keith & Jennifer Foskey,2026-02-11 04:17:22+00:00,processed,2026-02-27 21:22:35.987601+00:00
5,vPlN63MZUBY,YourCalvinist Live with Keith & Jennifer Foske...,2026-02-04 04:45:13+00:00,processed,2026-02-28 07:10:34.314457+00:00
6,qtdwQvLhHmk,YourCalvinist Live with Keith & Jennifer Foskey,2026-01-28 04:36:28+00:00,processed,2026-01-30 18:44:19.624576+00:00
7,qS8kiCWbZy0,YourCalvinist LIVE with Keith & Jennifer Foskey,2026-01-21 04:41:18+00:00,processed,2026-01-24 20:49:22.525565+00:00
8,F5dkh7xt-4A,YourCalvinist Podcast LIVE Q&A with Keith & Je...,2026-01-14 04:17:15+00:00,processed,2026-01-17 20:55:19.482509+00:00
9,sKlD7dukxdQ,YourCalvinist LIVE with Keith & Jen Foskey,2026-01-07 04:26:33+00:00,processed,2026-01-17 20:57:47.648739+00:00


## Check Missing Videos from Playlist

In [3]:
import sys
sys.path.insert(0, '/Users/spaceairmac/Dev/keith_foskey')

from app.youtube.playlist import get_playlist_video_ids
from app.settings import get_settings

settings = get_settings()

print("Fetching playlist videos...")
playlist_ids = get_playlist_video_ids()
playlist_len = len(playlist_ids) - 1

print(f"Playlist contains: {playlist_len} videos")

# Get database video IDs
with SessionLocal() as session:
    db_query = text("SELECT youtube_id FROM videos")
    db_videos = session.execute(db_query).fetchall()
    db_ids = set([row[0] for row in db_videos])

print(f"Database contains: {len(db_ids)} videos")
print(f"\nMissing from database: {playlist_len - len(db_ids)} videos\n")

# Find missing ones
missing = [vid for vid in playlist_ids if vid not in db_ids]

if missing:
    print("Missing video IDs:")
    for vid in missing:
        if vid != "4QpzXOyWDrE": # Confirmed with Keith to ignore this one since it's a duplicate
            print(f"  - {vid} (https://youtube.com/watch?v={vid})")
else:
    print("✓ All playlist videos are in the database!")

Fetching playlist videos...
Playlist contains: 76 videos
Database contains: 76 videos

Missing from database: 0 videos

Missing video IDs:


## 🔧 Manually Process a Single Video

Run a specific video through the ingestion pipeline directly.

In [ ]:
# Process a specific video manually
from app.ingest.pipeline import process_video

# The video ID to process
VIDEO_ID = "pRVSnDoj8tE"

# First, let's check and clear any stale lock on this job
with SessionLocal() as session:
    from sqlalchemy import text
    
    # Check current job status
    check_query = text("""
        SELECT youtube_id, status, attempts, locked_at, last_error 
        FROM ingest_jobs 
        WHERE youtube_id = :vid
    """)
    result = session.execute(check_query, {"vid": VIDEO_ID}).fetchone()
    
    if result:
        print(f"Current job status:")
        print(f"  Status: {result.status}")
        print(f"  Attempts: {result.attempts}")
        print(f"  Locked at: {result.locked_at}")
        print(f"  Last error: {result.last_error}")
        
        # Clear the stale lock if it's stuck in 'processing'
        if result.status == 'processing':
            reset_query = text("""
                UPDATE ingest_jobs 
                SET status = 'pending', locked_at = NULL 
                WHERE youtube_id = :vid
            """)
            session.execute(reset_query, {"vid": VIDEO_ID})
            session.commit()
            print("\n✓ Reset stale lock - job is now pending")
    else:
        print("No ingest job found for this video ID")

print("\n" + "="*60)
print(f"Processing video: {VIDEO_ID}")
print("="*60 + "\n")

# Run the pipeline directly
result = process_video(VIDEO_ID, skip_classification=False, verbose=True)

print("\n" + "="*60)
if result.success:
    print(f"✓ Success! Saved {result.questions_saved} Q&A items")
else:
    print(f"✗ Failed: {result.error}")